**1. List Comprehensions & Generator Expressions**

  In web dev, you use list comprehensions to map API responses. In ML, you use them for directory parsing, tokenization loops, and text cleaning.

**The ML Pattern: Inline Data Preprocessing**

In [5]:
# Batch processing strings for a text classification pipeline
raw_tweets = ["  BUY THIS NOW! #spam ", "Just dynamic programming things...", "  Free entry!! ",".."]

# Cleaner combination of stripping, lowercasing, and filtering out short text
clean_tokens = [tweet.strip().lower() for tweet in raw_tweets if len(tweet) > 5]

print(clean_tokens)

['buy this now! #spam', 'just dynamic programming things...', 'free entry!!']


**The ML Pattern: Generator Expressions for Memory Efficiency**

When dealing with millions of rows or heavy image tensors, loading a list comprehension into RAM can crash your machine (OOM - Out of Memory error). Instead, use a generator expression () to yield items lazily.

In [3]:
# 1. A generator function that yields paths one by one (uses 0 RAM)
def image_path_generator(total_images):
    for i in range(total_images):
        yield f"data/train/img_{i}.png"

# 2. Instantiate the generator
image_paths = image_path_generator(1000000)

# 3. Execution only happens as you loop through it
for path in image_paths:
    print(f"Processing: {path}")
    
    # In a real pipeline, you'd load your image here:
    # image = load_image(path) 
    
    break # Stopped immediately so it doesn't print 1 million lines!

Processing: data/train/img_0.png


**2. Generators (The <code>yield</code> keyword)**

Generators are the engine behind training loops. Because datasets can be hundreds of gigabytes, you cannot load them into memory all at once. You must stream them in small subsets called batches.

**The ML Pattern: Custom Batch Iterators**

This is structurally exactly how a PyTorch <code>DataLoader</code> behaves under the hood.

In [4]:
import numpy as np

# 1. Simulate 10GB of image arrays (using a smaller dummy shape for the example)
# Let's say 100 samples, each a 224x224x3 image
X_train = np.random.rand(100, 224, 224, 3)
y_train = np.random.randint(0, 2, size=100)

# 2. The batch generator (streams slices on-the-fly)
def batch_generator(X, y, batch_size=32):
    n_samples = len(X)
    for i in range(0, n_samples, batch_size):
        # NumPy slicing creates a "view", NOT a copy. Memory stays flat!
        yield X[i:i + batch_size], y[i:i + batch_size]

# 3. Use it in a training loop
for step, (X_batch, y_batch) in enumerate(batch_generator(X_train, y_train, batch_size=32)):
    print(f"Batch {step + 1} | X shape: {X_batch.shape} | y shape: {y_batch.shape}")

Batch 1 | X shape: (32, 224, 224, 3) | y shape: (32,)
Batch 2 | X shape: (32, 224, 224, 3) | y shape: (32,)
Batch 3 | X shape: (32, 224, 224, 3) | y shape: (32,)
Batch 4 | X shape: (4, 224, 224, 3) | y shape: (4,)


**3. Decorators**

Instead of managing web routes or authentication, decorators in ML are explicitly used for **caching heavy transformations, disabling gradient computations, and timing execution.**

**The ML Pattern: Disabling Gradients & Performance Optimization**

PyTorch utilizes decorators natively to tell the engine not to calculate mathematical slopes (gradients) during evaluation, which saves massive amounts of GPU memory.

In [5]:
import numpy as np

# 1. A simple custom decorator to simulate disabling tracking operations
def no_grad(func):
    def wrapper(*args, **kwargs):
        print("[System Check] Disabling tracking operations for performance...")
        result = func(*args, **kwargs)
        print("[System Check] Tracking operations re-enabled.\n")
        return result
    return wrapper

# 2. A mock model class with a forward pass
class SimpleModel:
    def forward(self, data):
        # Simulates a simple neural network layer computation (e.g., weights * data)
        weights = np.array([0.5, -0.2, 0.1])
        return np.dot(data, weights)

# 3. Apply the decorator to the evaluation function
@no_grad
def predict_unlabeled_data(model, data):
    return model.forward(data)

# 4. Run the code
if __name__ == "__main__":
    # Generate mock unlabeled input features for 3 samples
    X_unlabeled = np.random.rand(3, 3)
    model = SimpleModel()
    
    # Run prediction
    predictions = predict_unlabeled_data(model, X_unlabeled)
    print(f"Model Predictions:\n{predictions}")

[System Check] Disabling tracking operations for performance...
[System Check] Tracking operations re-enabled.

Model Predictions:
[0.10453214 0.14597252 0.41125998]


**The ML Pattern: Registering Custom Components**

If you study production ML architectures (like Hugging Face Transformers or AllenNLP), you will see decorators used to auto-register custom model layers or feature metrics into a global registry dynamically.

**The ML Pattern: Hyperparameter Passthrough**

* **The Setup**: Instead of writing 50 settings in your main function, you pass one config dictionary.

* **The Funnel**: The outer pipeline uses `**kwargs` to collect any incoming settings without needing to name them.

* **The Forward**: The `**` syntax unzips that dictionary directly into the underlying machine learning models.


In [8]:
def train_model(learning_rate=0.01, epochs=10):
    print(f"LR: {learning_rate}, Epochs: {epochs}")

# **kwargs catches and forwards any settings
def ml_pipeline(data, **kwargs):
    train_model(**kwargs)

# Pass settings directly
config = {"learning_rate": 0.05, "epochs": 20}
ml_pipeline("data.csv", **config)

LR: 0.05, Epochs: 20


**5. Object-Oriented Programming (OOP)**

ML repositories are not strictly functional; they are highly structural. You will see a repetitive reliance on **subclass inheritance**, specifically building custom classes derived from a parent blueprint.

**The ML Pattern: Implementing Interface Overrides**

Whether you are writing a custom Scikit-Learn transformer or a PyTorch model layer, you inherit from a base class and override specific lifecycle hooks (like `fit`/`transform` or `__init__`/`forward`).


In [9]:
# The Parent Blueprint provided by a library
class BaseModel:
    def forward(self, x):
        raise NotImplementedError("You must write your own forward pass!")

# Your Custom Subclass (inheriting the blueprint)
class MyNeuralNetwork(BaseModel):
    # Overriding the blueprint's hook with your custom logic
    def forward(self, x):
        return x * 2  # Your specific ML layer logic here

# Run the model
model = MyNeuralNetwork()
print(model.forward(5))  # Output: 10

10


In [10]:
import numpy as np

class CustomMinMaxScaler:
    def __init__(self):
        self.min_ = None
        self.max_ = None
        
    def fit(self, X):
        # 1. Save the lowest and highest numbers from your training data
        self.min_ = np.min(X)
        self.max_ = np.max(X)
        return self  # Allows chaining like .fit().transform()
        
    def transform(self, X):
        # 2. Scale the numbers so they all fall strictly between 0 and 1
        return (X - self.min_) / (self.max_ - self.min_)

# --- How to use it in a pipeline ---

# Mock training data (e.g., house prices in thousands)
X_train = np.array([100, 200, 300, 400, 500])

# Initialize your custom scaler tool
scaler = CustomMinMaxScaler()

# Fit (learn the min/max) and Transform (scale the data) all at once
scaled_features = scaler.fit(X_train).transform(X_train)

print(f"Original Data: {X_train}")
print(f"Scaled Data:   {scaled_features}")


Original Data: [100 200 300 400 500]
Scaled Data:   [0.   0.25 0.5  0.75 1.  ]



| Code Structure | Full-Stack Web Dev Use Case | Machine Learning Use Case |
| :--- | :--- | :--- |
| **Generators** | Streaming file uploads or paginated API results | Streaming data chunks (batches) without running out of RAM |
| **Decorators** | Authentication checks, logging, and routing | Tracking training metrics, timing, or disabling gradients |
| **`**kwargs`** | Forwarding HTML attributes down a component hierarchy | Passing structural hyperparameter dictionaries down to base algorithms |
| **OOP Inheritance** | Creating database models or controller routing layers | Overriding lifecycles like custom feature transformers or Neural Network layer steps |
